# Hosting your own AI Chat App - with LibreChat and Ollama

Want your own open-source "ChatGPT"? In this notebook, we set up [LibreChat](https://www.librechat.ai) as the chat interface and connect it to open language models served by Ollama - running entirely on your own machine.

**LibreChat** is an open-source chat application in the style of ChatGPT. It works with many model providers - OpenAI, Anthropic, Google, and any service with an OpenAI-compatible API, such as Ollama. It supports multiple users, conversation history, and switching between models within a conversation.

In [ ]:
from ai_dojo import show

In [ ]:
show.github_repo("https://github.com/danny-avila/LibreChat")

## What You Need

- [**Docker**](https://docs.docker.com/get-docker/) with Docker Compose (on macOS and Windows: Docker Desktop)
- [**Git**](https://git-scm.com/downloads)
- [**Ollama**](https://ollama.com/download), running on your machine with at least one chat model (see [Serving Open LLMs with Ollama](lm-intro-ollama.ipynb))

LibreChat runs in Docker containers, while Ollama runs directly on your machine. This matters on a Mac: only a natively installed Ollama can use the GPU of Apple chips.

## Step 1: Check Ollama

Let's make sure Ollama is running, and see which models are available:

In [ ]:
import ollama
from ai_dojo.lm import get_ollama_url

client = ollama.Client(host=get_ollama_url())
[m.model for m in client.list().models]

If the list is empty, pull a chat model first, for example:

In [ ]:
show.command("ollama pull qwen3.5:4b")

## Step 2: Download LibreChat

Clone the LibreChat repository into a directory of your choice, and create the settings file `.env` from its template:

In [ ]:
show.command("git clone https://github.com/danny-avila/LibreChat.git")

In [ ]:
show.command("cd LibreChat && cp .env.example .env")

On Windows, use `copy` instead of `cp`.

The defaults in `.env` are fine for a private installation on your own machine: LibreChat generates the secrets it needs for logins on first start. Before you make LibreChat available to other people, set your own persistent values for `CREDS_KEY`, `CREDS_IV`, `JWT_SECRET` and `JWT_REFRESH_SECRET` - see the [LibreChat documentation](https://www.librechat.ai/docs/configuration/dotenv).

One value is needed right away: the Docker setup includes an admin panel, which refuses to start without a session secret. Let's generate one:

In [ ]:
import secrets

secrets.token_hex(32)

Copy the value into the line `ADMIN_PANEL_SESSION_SECRET=` in `.env`:

```
ADMIN_PANEL_SESSION_SECRET=<value from above>
```

## Step 3: Connect LibreChat to Ollama

Ollama offers an OpenAI-compatible API, so LibreChat can use it as a **custom endpoint**. Create a file `librechat.yaml` in the `LibreChat` directory with the following content:

```yaml
version: 1.3.17
cache: true

endpoints:
  custom:
    - name: 'Ollama'
      apiKey: 'ollama'
      baseURL: 'http://host.docker.internal:11434/v1/'
      models:
        default: ['qwen3.5:4b']
        fetch: true
      titleConvo: true
      titleModel: 'current_model'
      summarize: false
      modelDisplayLabel: 'Ollama'
      dropParams: ['stop', 'user', 'frequency_penalty', 'presence_penalty']
```

What these settings do:

- `baseURL`: LibreChat runs inside a Docker container. From there, `host.docker.internal` is the address of your own machine, where Ollama listens on port 11434.
- `apiKey`: Ollama does not check API keys, but LibreChat requires a value.
- `models`: with `fetch: true`, LibreChat offers all models you have pulled with Ollama. The `default` list is the fallback in case fetching fails.
- `titleConvo`, `titleModel`: the current model generates a title for each conversation.
- `dropParams`: request parameters that LibreChat should not send to Ollama.

The [LibreChat documentation](https://www.librechat.ai/docs/configuration/librechat_yaml) describes all options.

Next, tell Docker Compose to mount this file into the LibreChat container. Create a file `docker-compose.override.yaml` in the same directory:

```yaml
services:
  api:
    volumes:
      - type: bind
        source: ./librechat.yaml
        target: /app/librechat.yaml
```

Keeping local changes in this override file leaves LibreChat's own `docker-compose.yml` untouched, which keeps updates simple.

### Linux only: make Ollama reachable from Docker

On macOS and Windows, Docker Desktop connects `host.docker.internal` to your machine out of the box. On Linux, Ollama listens only on `127.0.0.1` by default, which containers cannot reach. Configure the Ollama service to listen on all network interfaces:

In [ ]:
show.command("sudo systemctl edit ollama")

Add these lines, save, and restart Ollama with `sudo systemctl restart ollama`:

```
[Service]
Environment="OLLAMA_HOST=0.0.0.0"
```

This also makes Ollama reachable from other machines in your network. If you don't want that, block port 11434 in your firewall.

## Step 4: Start LibreChat

In the `LibreChat` directory, start all services in the background:

In [ ]:
show.command("docker compose up -d")

The first start downloads several container images and takes a few minutes. Besides LibreChat itself, Docker starts a few supporting services, such as the MongoDB database that stores users and conversations.

Let's check whether LibreChat responds:

In [ ]:
import requests

librechat_url = "http://localhost:3080"
requests.get(librechat_url, timeout=5).status_code

A status code of `200` means LibreChat is up. If the connection is refused, wait a minute and try again, or look at the logs with `docker compose logs -f api`.

## Step 5: Chat

1. Open [http://localhost:3080](http://localhost:3080) and **sign up**. The account only exists in your local database.
2. In the model selector, choose the endpoint **Ollama** and one of your models.
3. Start chatting!

## Managing LibreChat

Run these commands in the `LibreChat` directory:

| Task | Command |
|------|---------|
| Stop LibreChat | `docker compose down` |
| Start it again | `docker compose up -d` |
| Show the logs | `docker compose logs -f api` |
| Update to the latest version | `git pull`, then `docker compose pull` and `docker compose up -d` |

If a model you pulled later does not show up in the model selector, restart LibreChat with `docker compose restart api`.

## Exercise

1. Ask a small and a larger model the same question, and compare their answers.
2. Open the parameters panel and change the **temperature**. How do the answers change?

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2026 [Christian Staudt](https://clstaudt.me)_